# Retinal Vessel Segmentation Training Notebook (Google Colab / Kaggle)
Senior Capstone: Retinal Vascular Biometric Authentication System (Week 3 & Week 4)
- Architecture: U-Net with Pretrained ResNet-34 Encoder (segmentation_models_pytorch)
- Loss Function: Combined Loss (BCEWithLogitsLoss + DiceLoss)
- Input: 512x512 RGB / Output: Binary Vessel Mask

In [ ]:
# 1. Install Dependencies and Check GPU
!nvidia-smi
!pip install -q segmentation-models-pytorch albumentations opencv-python-headless scikit-learn matplotlib tqdm

In [ ]:
# 2. Verify GPU Availability
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

In [ ]:
# 3. Run Training Pipeline via train_pipeline.py
from notebooks.train_pipeline import run_training

run_training(
    data_dir='data',
    epochs=30,
    batch_size=4,
    learning_rate=1e-4,
    checkpoint_path='models/best_unet_model.pth',
    plot_path='results/training_curves.png',
    patience=8
)

In [ ]:
# 4. Evaluate Metrics on Test Sets (DRIVE, HRF, STARE)
from evaluation.evaluate_vessel import run_full_evaluation

results = run_full_evaluation(
    model_or_checkpoint='models/best_unet_model.pth',
    data_dir='data',
    output_report_path='results/vessel_evaluation_report.md',
    use_tta=True
)

In [ ]:
# 5. Qualitative Visual Comparison Samples
import importlib.util
from PIL import Image
import matplotlib.pyplot as plt

spec = importlib.util.spec_from_file_location('evaluate_visual', 'notebooks/02_evaluate_visual.py')
vis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vis)

vis.generate_visual_comparison_samples(
    model_or_checkpoint='models/best_unet_model.pth',
    data_dir='data',
    output_path='results/vessel_prediction_samples.png',
)

img = Image.open('results/vessel_prediction_samples.png')
plt.figure(figsize=(16, 12))
plt.imshow(img)
plt.axis('off')
plt.show()